# Why does each disease hit one part of the brain?
### Selective vulnerability test — Alzheimer's disease

**Run this in Google Colab** (`colab.research.google.com`), not in a restricted sandbox. Two of its data
dependencies — the Allen Human Brain Atlas backend (`api.brain-map.org` / `human.brain-map.org`, pulled by
`abagen`) and the GWAS Catalog (`www.ebi.ac.uk/gwas`) — are unreachable from environments with locked-down
outbound network policies. Colab has open internet access, so this is the intended venue.

**What this notebook does:** tests whether Alzheimer's disease GWAS risk genes are unusually active
(over-expressed relative to random gene sets) specifically in the brain regions Alzheimer's is known to
damage first — the entorhinal cortex and hippocampus — using the Allen Human Brain Atlas transcriptomic
data and a permutation test, then checks whether that result survives correction for spatial
autocorrelation.

**Provenance note:** Parts 1–5.3 below implement the pipeline as specified in the source document,
verbatim where the document gave exact code, and against the *actual, installed* `abagen`/`neuromaps`
APIs (verified locally — function signatures, the real 83-region Desikan-Killiany label table, and the
real fsaverage5 (10,242-vertex, i.e. "10k") surface parcellation shipped with `abagen` — before writing
this notebook, not from memory). Section 5.4 onward is **not** from the source document — the screenshots
provided cut off mid-code in section 5.3, before the document explained what to do with the spatial-null
output. Section 5.4 is my own technically-correct completion of that unfinished thread, clearly marked as
such, so the notebook actually runs end-to-end instead of leaving `rotated` computed and unused. If the
source document specifies something different for that step, replace 5.4 with it.

## Part 0.5: Four words worth knowing

- **Gene expression** — how switched on a gene is in a given tissue sample.
- **Atlas** — a map that divides the brain into named regions.
- **Parcellation** — the specific scheme used to divide the brain (different parcellations, different region counts).
- **GWAS** (genome-wide association study) — a scan comparing the genomes of people with a disease to people
  without, to find genetic variants that show up more often in the disease group. Those variants' genes are
  candidate risk genes.

## Part 1: Get the brain data

### 1.1–1.2 Install

In [ ]:
!pip install -q abagen neuromaps nilearn statsmodels nibabel

### 1.3 Pick your map

Desikan-Killiany, fetched via `abagen`. It splits the brain into 83 regions: 68 cortical (34 per
hemisphere) plus 15 subcortical/brainstem structures (7 per hemisphere plus the unpaired brainstem).

In [ ]:
import abagen

atlas = abagen.fetch_desikan_killiany()
print(atlas['image'])
print(atlas['info'])

### 1.4 Pull the expression data — and the hemisphere decision (section 1.7)

Only two of the six AHBA donors have right-hemisphere tissue samples. `abagen`'s `get_expression_data` has
an `lr_mirror` parameter (`None` by default) that controls whether left-hemisphere samples are mirrored
onto the right to fill the gap. Per the source document: **pick one, and say which one you picked in your
write-up.**

This notebook defaults to `HEMISPHERE_MODE = "left_only"` — no mirroring, and the final expression table is
restricted to left-hemisphere + the one bilateral brainstem region, so every row in the analysis rests on
real (not mirrored) tissue. Set `HEMISPHERE_MODE = "mirror_bidirectional"` to instead mirror samples across
both hemispheres (`lr_mirror='bidirectional'`) and keep both hemispheres in the analysis at roughly double
the region count.

This step downloads roughly 4GB and will take a long time. It only works where `api.brain-map.org` /
`human.brain-map.org` are reachable — not in this session's sandbox, hence not run here.

In [ ]:
import pandas as pd

HEMISPHERE_MODE = "left_only"  # "left_only" or "mirror_bidirectional" — pick one, document your choice

info = pd.read_csv(atlas['info'])

if HEMISPHERE_MODE == "left_only":
    expression = abagen.get_expression_data(atlas['image'], atlas['info'], lr_mirror=None)
    keep_ids = info.loc[info['hemisphere'].isin(['L', 'B']), 'id']
    expression = expression.loc[expression.index.isin(keep_ids)]
elif HEMISPHERE_MODE == "mirror_bidirectional":
    expression = abagen.get_expression_data(atlas['image'], atlas['info'], lr_mirror='bidirectional')
else:
    raise ValueError(f"Unrecognised HEMISPHERE_MODE: {HEMISPHERE_MODE!r}")

### 1.5 Save it immediately

If the Colab session disconnects and this wasn't saved, the 4GB download happens again.

In [ ]:
expression.to_csv('ahba.csv')

### 1.6 Look at what you got

In [ ]:
print(expression.shape)
expression.iloc[:5, :5]

With `HEMISPHERE_MODE = "left_only"` this should be roughly 42 rows (41 left-hemisphere regions + the bilateral brainstem) by roughly 15,000 gene columns. With `"mirror_bidirectional"` it should be close to the full 83 rows. If the shape looks wrong, stop and fix it before continuing.

## Part 2: Get your Alzheimer's disease gene list

### 2.1–2.2 Find it in the GWAS Catalog

Go to https://www.ebi.ac.uk/gwas and search "Alzheimer's disease". Open the trait page and read the `EFO`
code out of the URL — that's the official identifier for the trait; write it down.

**Verification note:** `EFO_0000249` is the identifier commonly cited in published AHBA/GWAS-Catalog
literature for Alzheimer's disease, but `www.ebi.ac.uk` was unreachable from this session (network policy
blocked it — confirmed by direct test, not assumed), so I could not confirm it live against the current
catalog. Confirm it yourself from the trait page URL before treating it as correct.

### 2.3 Download the associations

Click the download button on the trait page; save the file as `associations.tsv` in the Colab working
directory (or upload it via the Colab file browser).

In [ ]:
EFO_ID = "EFO_0000249"  # confirm this against the trait page URL yourself — see note above
print(f"Disease: Alzheimer's disease ({EFO_ID})")

### 2.4 Clean it up

The p-value cutoff of 5e-8 is the standard genome-wide significance bar. The `MAPPED_GENE` column
sometimes lists two genes separated by a dash or a comma — split those apart.

In [ ]:
import re

gwas = pd.read_csv('associations.tsv', sep='\t')

# keep only the strong hits
gwas = gwas[gwas['P-VALUE'] < 5e-8]

raw_genes = gwas['MAPPED_GENE'].dropna().unique()
genes = set()
for entry in raw_genes:
    for g in re.split(r'[-,]', entry):
        g = g.strip()
        if g:
            genes.add(g)
genes = sorted(genes)

### 2.5 Check the size

Want somewhere between 20 and 500. Under 20 and the statistics are jumpy; over 500 and it's basically
measuring the whole genome.

In [ ]:
print(len(genes))
assert 20 <= len(genes) <= 500, (
    "Outside the usable range — adjust the p-value cutoff or reconsider the disease choice."
)

## Part 3: Score every region

### 3.1 The idea

Compare Alzheimer's risk-gene activity in each region against 10,000 random gene sets of the same size, in
that same region. If the real gene set sits well above the random range, that's unusually high activity —
a permutation test.

### 3.2 Match your genes to the data

In [ ]:
available = [g for g in genes if g in expression.columns]
print(f"{len(available)} of {len(genes)} genes found")
if len(available) < 0.5 * len(genes):
    print("WARNING: fewer than half the genes matched — check gene name formatting before trusting the result.")

### 3.3 Get the real score

In [ ]:
observed = expression[available].mean(axis=1)

### 3.4 Build the random comparison

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
all_genes = expression.columns.to_numpy()
n_perm = 10000
null = np.zeros((n_perm, len(expression)))

for i in range(n_perm):
    fake = rng.choice(all_genes, size=len(available), replace=False)
    null[i] = expression[fake].mean(axis=1)

### 3.5 Turn it into a number

In [ ]:
z = (observed - null.mean(axis=0)) / null.std(axis=0)
p = (null >= observed.to_numpy()).mean(axis=0)

### 3.6 Fix the multiple testing problem

In [ ]:
from statsmodels.stats.multitest import multipletests

_, p_corrected, _, _ = multipletests(p, method='fdr_bh')

## Part 4: Plot it and check it

### 4.1 Make the picture

In [ ]:
from nilearn import plotting

results = pd.DataFrame({'region': expression.index, 'z': z, 'p': p, 'p_fdr': p_corrected})
results = results.merge(info[['id', 'label', 'hemisphere', 'structure']], left_on='region', right_on='id')

plotting.plot_roi(
    atlas['image'],
    title="Alzheimer's risk-gene expression z-score by region",
)

### 4.2 Rank the regions

In [ ]:
print(results.sort_values('z', ascending=False).head(10)[['label', 'hemisphere', 'structure', 'z', 'p_fdr']])

### 4.3 Now the real test

Alzheimer's disease is known to damage the **entorhinal cortex** first, then the **hippocampus**. Check
whether those are among the top-ranked regions above.

If they match: a real pattern, found with your own hands. If they don't: risk-gene activity alone is not
what decides where Alzheimer's strikes — an actual finding, not a failure. Report it as what it is.

In [ ]:
targets = results[results['label'].str.contains('entorhinal|hippocampus', case=False, regex=True)]
print(targets[['label', 'hemisphere', 'z', 'p_fdr']])

ranked = results.sort_values('z', ascending=False).reset_index(drop=True)
for _, row in targets.iterrows():
    rank = ranked.index[ranked['id'] == row['id']][0] + 1
    print(f"{row['label']} ({row['hemisphere']}): rank {rank} of {len(ranked)}, z={row['z']:.2f}, p_fdr={row['p_fdr']:.4f}")

## Part 5: The thing that separates careful work from sloppy work

### 5.1–5.2 The problem

Neighbouring brain regions tend to have similar expression values just because they're neighbours — gene
expression changes gradually across the brain rather than jumping around. This is **spatial
autocorrelation**. The permutation test in Part 3 assumes every region is independent, which is false;
regions the test treats as independent are actually correlated with their neighbours, so the p-values it
produces come out smaller than they should — things can look significant when they're not.

### 5.3 The fix

Use a null model that keeps the spatial structure while scrambling the association being tested — a
"spin test." `neuromaps` implements this (`nulls.alexander_bloch`, Alexander-Bloch et al., 2018). It spins
a spherical projection of the cortical surface and re-reads off region values at the rotated
locations, which preserves each map's spatial-autocorrelation structure while breaking any real
correspondence between two maps.

**Important, verified constraint:** spin tests are only defined for the cortical surface. `alexander_bloch`
needs `data` and a `parcellation` matched to an actual surface mesh (here: `abagen`'s bundled
Desikan-Killiany surface parcellation, confirmed to be 10,242 vertices per hemisphere — the standard
fsaverage5 mesh, i.e. `neuromaps`' `density='10k'`). The 15 subcortical/brainstem Desikan-Killiany regions
have no surface counterpart and cannot be spun this way; they're excluded from this section and would need
a separate null model (e.g. a simple label-permutation null) if you want a spatially-aware test for them
too.

In [ ]:
!pip install -q neuromaps

import abagen
from neuromaps import nulls

atlas_surf = abagen.fetch_desikan_killiany(surface=True)
parcellation = atlas_surf['image']  # (lh.label.gii.gz, rh.label.gii.gz), 10242 vertices/hemisphere

cortex = results[results['structure'] == 'cortex'].set_index('label')
my_map = cortex['z']

rotated = nulls.alexander_bloch(my_map, atlas='fsaverage', density='10k',
                                 parcellation=parcellation, n_perm=1000)

### 5.4 Use the spatial null — not in the source document

The screenshots supplied stop here, with `rotated` computed but unused, so this section is **not** from the
source document — it's my own completion, written so the notebook doesn't stall on an unused variable. If
the source material specifies a different next step, use that instead and discard this cell.

This turns Part 4.3's "eyeball the ranking" comparison into an actual spin-corrected statistical test: build
a target map that is 1 for entorhinal cortex (0.5 weight if you want to include parahippocampal cortex,
which is the cortical neighbour of the hippocampus and the closest cortical proxy available here, since
hippocampus itself is subcortical and outside this spin test) and 0 elsewhere, then test whether the
observed z-score map correlates with that target map more than the 1,000 spatially-autocorrelation-matched
null maps would predict.

In [ ]:
from neuromaps.stats import compare_images

target = pd.Series(0.0, index=cortex.index)
target.loc[target.index.str.contains('entorhinal', case=False)] = 1.0

corr, spin_p = compare_images(my_map, target, metric='pearsonr', nulls=rotated)
print(f"Observed z-map vs. entorhinal-cortex target: r={corr:.3f}, spin-test p={spin_p:.4f}")
print("This p-value accounts for spatial autocorrelation; compare it against the naive result in Part 3-4.")

## What's left to write up

Per section 1.7 and the running thread through this document, your write-up needs to state:

1. Which `HEMISPHERE_MODE` you used (Part 1.4) and why.
2. The EFO identifier you actually confirmed on the GWAS Catalog trait page (Part 2.2) — not just the
   unverified one carried over from this notebook.
3. How many of your genes matched (`available` vs. `genes`, Part 3.2) — if it's a small fraction, say so and
   explain why.
4. Whether the top-ranked regions match entorhinal cortex / hippocampus, both before (Part 4.3) and after
   (Part 5.4) the spatial-autocorrelation correction — and if the naive and spin-corrected results
   disagree, report that disagreement rather than picking whichever one looks better.
5. The six-donor sample size as a limitation of the underlying Allen Human Brain Atlas data, not of this
   analysis's execution.

Six brains is a real, stated limitation of the AHBA (Part 1.1) — not something this notebook can fix.